# 07 - Generative posterior sampling & uncertainty quantification

Lecture section: 4.9-4.10, 5.2  |  Spine term this tutorial changes: $R$ is a learned
**generative** prior, and instead of one $\hat{x}$ we **sample the posterior** $p(x\mid y)$

$$x \sim p(x\mid y)\ \propto\ \underbrace{\exp\!\big(-D(Ax,y)\big)}_{\text{data fidelity}}\ \underbrace{p_\theta(x)}_{\text{generative prior}}$$

Every method so far returned **one** reconstruction. But an ill-posed problem has **many**
solutions consistent with $y$. A generative (diffusion) prior lets us **sample** several of them:
their average is the posterior mean (the minimum-MSE estimate), and their spread is a per-pixel
**uncertainty map**.

*Note on compute:* diffusion posterior sampling needs hundreds of denoiser evaluations. On the
heavy CT/Radon operator that is GPU territory, so for a CPU demo we keep the **same algorithm**
but use a light, decomposable operator -- random inpainting of a natural image -- where it runs in
seconds. The idea is identical for CT/MRI.

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt

import tutorial_common as tc
import deepinv as dinv

tc.set_seed()
N = 64

deepinv 0.4.1 | torch 2.9.1 | device cpu


## Setup: a natural image, random inpainting, and a diffusion (DPS) sampler

The generative prior is a pretrained **DRUNet** denoiser used as the diffusion score; **DPS**
(Diffusion Posterior Sampling) turns it into a posterior sampler for our forward operator $A$.

In [2]:
x = dinv.utils.load_example("butterfly.png", img_size=N, device=tc.DEVICE)     # (1,3,N,N) color
phys = dinv.physics.Inpainting(
    img_size=(3, N, N), mask=0.4,                                              # keep 40% of pixels
    noise_model=dinv.physics.GaussianNoise(sigma=0.03), device=tc.DEVICE,
)
y = phys(x)
denoiser = dinv.models.DRUNet(pretrained="download", device=tc.DEVICE)         # the generative prior
sampler = dinv.sampling.DPS(
    denoiser=denoiser, schedule="vp", num_steps=150, weight=1.0, alpha=1.0,
    verbose=False, device=tc.DEVICE, dtype=torch.float32,
    rng=torch.Generator(device=tc.DEVICE), minus_one_one=False,
)


# small helper to show color (3ch) or gray (1ch) tensors
def imshow(ax, img, title):
    a = img.detach().squeeze(0).clamp(0, 1).cpu()
    ax.imshow(a.permute(1, 2, 0).numpy() if a.shape[0] == 3 else a.squeeze(0).numpy(), cmap="gray")
    ax.set_title(title, color=tc.PALETTE["slate"])
    ax.axis("off")

## Draw several posterior samples

Each call with a different seed returns a **different** plausible reconstruction -- all consistent
with the same measurements $y$.

In [3]:
n_samples = 4
samples = []
for sd in range(1, n_samples + 1):
    with torch.no_grad():
        out = sampler(y.clone(), phys, seed=sd)
    s = out[0] if isinstance(out, (tuple, list)) else out
    samples.append(s.float())
    print(f"sample {sd}: PSNR {tc.psnr(s.float(), x):.2f} dB")
S = torch.stack(samples)  # (n_samples, 1, 3, N, N)

sample 1: PSNR 24.76 dB


sample 2: PSNR 24.70 dB


sample 3: PSNR 24.76 dB


sample 4: PSNR 24.68 dB


## The samples differ -- the reconstruction is a DISTRIBUTION, not a point

In [4]:
fig, axs = plt.subplots(1, n_samples + 2, figsize=(2.4 * (n_samples + 2), 3.0))
imshow(axs[0], x, "x (truth)")
imshow(axs[1], y, "masked y")
for i in range(n_samples):
    imshow(axs[2 + i], samples[i], f"sample {i + 1}\nPSNR {tc.psnr(samples[i], x):.1f} dB")
fig.suptitle("Posterior samples: several plausible x for the same y", color=tc.PALETTE["slate"])
fig.tight_layout()
fig.savefig(tc.FIG_DIR / "07_posterior_samples.png", dpi=150)
plt.close(fig)
print("saved 07_posterior_samples.png")

saved 07_posterior_samples.png


## Posterior mean (MMSE) and the uncertainty map

Averaging the samples gives the **posterior mean** -- the minimum-MSE estimate, typically at least
as good as any single sample. The per-pixel **standard deviation** across samples is an
**uncertainty map**: it is large exactly where the samples disagree (missing regions, fine
texture, edges) -- and it tracks the true error.

In [5]:
mean = S.mean(0)                                  # posterior mean (MMSE)
std = S.std(0).mean(1, keepdim=True)              # per-pixel std, averaged over color channels
err = (mean - x).abs().mean(1, keepdim=True)      # actual reconstruction error
best = max(tc.psnr(s, x) for s in samples)
print(f"posterior mean PSNR {tc.psnr(mean, x):.2f} dB   (best single sample {best:.2f} dB)")

fig, axs = plt.subplots(1, 4, figsize=(13, 3.3))
imshow(axs[0], x, "x (truth)")
imshow(axs[1], mean, f"posterior mean (MMSE)\nPSNR {tc.psnr(mean, x):.1f} dB")
for ax, m, title in [(axs[2], std, "uncertainty: per-pixel std"),
                     (axs[3], err, "actual error |mean - x|")]:
    h = ax.imshow(m.squeeze().cpu().numpy(), cmap="magma")
    ax.set_title(title, color=tc.PALETTE["slate"])
    ax.axis("off")
    fig.colorbar(h, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("Posterior mean + uncertainty (the std map tracks the true error)",
             color=tc.PALETTE["slate"])
fig.tight_layout()
fig.savefig(tc.FIG_DIR / "07_uncertainty.png", dpi=150)
plt.close(fig)
print("saved 07_uncertainty.png")

posterior mean PSNR 25.97 dB   (best single sample 24.76 dB)


saved 07_uncertainty.png


## Takeaway

A generative prior turns reconstruction into **posterior sampling**: instead of one image we get a
whole distribution. Its mean is the MMSE estimate, and its spread is a per-pixel uncertainty map
that flags where the reconstruction is least trustworthy -- exactly what high-stakes imaging needs.
The same algorithm applies to CT/MRI; the heavy forward operators just call for a GPU.